# NeuroRouter API Testing

Testing the NeuroRouter AWS backend with a real API key.
This verifies that:
1. API key authentication works
2. LLM inference works (Groq backend via AWS Lambda)
3. Token usage is recorded in DynamoDB
4. Dashboard reflects the usage

In [1]:
import openai
import json

# NeuroRouter AWS backend
API_BASE = "https://u87jos3lg5.execute-api.ap-south-1.amazonaws.com/dev"

# Create an API key from the dashboard first (http://localhost:3000/dashboard/api-keys)
# Then paste it here:"neurorouter_3Tupa7AtRUamP", neurorouter_k69hTSo1iVFfN
API_KEY = "neurorouter_CbmLIKlErviQW"  #neurorouter_Kks8EyqGq813P e.g. "neurorouter_xxxxxxxxx"#neurorouter_21cf0pza3OZUb

client = openai.OpenAI(
    base_url=f"{API_BASE}/v1",
    api_key=API_KEY,
)

print(f"Connected to: {API_BASE}")
print(f"API Key: {API_KEY[:20]}...")

Connected to: https://u87jos3lg5.execute-api.ap-south-1.amazonaws.com/dev
API Key: neurorouter_CbmLIKlE...


## Test 1: List Available Models

In [2]:
models = client.models.list()
print(f"Available models: {len(models.data)}")
for m in models.data:
    print(f"  - {m.id} (owned by: {m.owned_by})")

Available models: 5
  - llama-3.3-70b-versatile (owned by: groq)
  - llama-3.1-8b-instant (owned by: groq)
  - llama-3.1-70b-versatile (owned by: groq)
  - mixtral-8x7b-32768 (owned by: groq)
  - gemma2-9b-it (owned by: groq)


## Test 2: Custom Prompt — llama-3.3-70b-versatile
Enter your own prompt below. This sends it to the `llama-3.3-70b-versatile` model and shows the response + token usage.

In [3]:
# Enter your custom prompt for llama-3.3-70b-versatile
prompt = input("Enter your prompt for llama-3.3-70b-versatile: ")

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "user", "content": prompt}
    ],
    max_tokens=500,
)

print(f"Model: llama-3.3-70b-versatile")
print(f"Prompt: {prompt}")
print(f"\nAI Response:")
print(response.choices[0].message.content)
print(f"\nTokens: prompt={response.usage.prompt_tokens}, completion={response.usage.completion_tokens}, total={response.usage.total_tokens}")
print("\n--- Check dashboard: this usage should reflect under llama-3.3-70b-versatile ---")

Model: llama-3.3-70b-versatile
Prompt: what is python

AI Response:
**Python Overview**

Python is a high-level, interpreted programming language that is widely used for various purposes such as:

* Web development
* Data analysis and science
* Artificial intelligence and machine learning
* Automation
* Scientific computing
* Education

**Key Features of Python**
-------------------------

* **Easy to learn**: Python has a simple syntax and is relatively easy to learn, making it a great language for beginners.
* **High-level language**: Python abstracts away many low-level details, allowing developers to focus on the logic of their program without worrying about memory management and other details.
* **Interpreted language**: Python code is executed line-by-line, making it easier to debug and test.
* **Object-oriented**: Python supports object-oriented programming (OOP) concepts such as classes, objects, inheritance, and polymorphism.
* **Large standard library**: Python has a vast col

## Test 3: Custom Prompt — mixtral-8x7b-32768
Enter your own prompt below. This sends it to the `mixtral-8x7b-32768` model and shows the response + token usage.

In [4]:
# Enter your custom prompt for mixtral-8x7b-32768
prompt = input("Enter your prompt for mixtral-8x7b-32768: ")

response = client.chat.completions.create(
    model="mixtral-8x7b-32768",
    messages=[
        {"role": "user", "content": prompt}
    ],
    max_tokens=500,
)

print(f"Model: mixtral-8x7b-32768")
print(f"Prompt: {prompt}")
print(f"\nAI Response:")
print(response.choices[0].message.content)
print(f"\nTokens: prompt={response.usage.prompt_tokens}, completion={response.usage.completion_tokens}, total={response.usage.total_tokens}")
print("\n--- Check dashboard: this usage should reflect under mixtral-8x7b-32768 ---")

Model: mixtral-8x7b-32768
Prompt: what is python

AI Response:
**Python** is a high-level, interpreted programming language that is widely used for various purposes such as:

* **Web Development**: Creating web applications and services using popular frameworks like Django and Flask.
* **Data Analysis and Science**: Analyzing and visualizing data using libraries like Pandas, NumPy, and Matplotlib.
* **Artificial Intelligence and Machine Learning**: Building AI and ML models using libraries like TensorFlow and Scikit-learn.
* **Automation**: Automating tasks and processes using Python's extensive library of modules and tools.
* **Education**: Teaching programming concepts and principles due to its simplicity and readability.

**Key Features of Python:**

1. **Easy to Learn**: Python has a simple syntax and is relatively easy to learn, making it a great language for beginners.
2. **High-Level Language**: Python is a high-level language, meaning it abstracts away many low-level details, a

## Test 4: Custom Prompt — gemma2-9b-it
Enter your own prompt below. This sends it to the `gemma2-9b-it` model and shows the response + token usage.

In [5]:
# Enter your custom prompt for gemma2-9b-it
prompt = input("Enter your prompt for gemma2-9b-it: ")

response = client.chat.completions.create(
    model="gemma2-9b-it",
    messages=[
        {"role": "user", "content": prompt}
    ],
    max_tokens=500,
)

print(f"Model: gemma2-9b-it")
print(f"Prompt: {prompt}")
print(f"\nAI Response:")
print(response.choices[0].message.content)
print(f"\nTokens: prompt={response.usage.prompt_tokens}, completion={response.usage.completion_tokens}, total={response.usage.total_tokens}")
print("\n--- Check dashboard: this usage should reflect under gemma2-9b-it ---")

Model: gemma2-9b-it
Prompt: what is python

AI Response:
**Python** is a high-level, interpreted programming language that is widely used for various purposes such as:

* **Web development**: building web applications and web services
* **Data analysis**: working with data, statistics, and machine learning
* **Artificial intelligence**: building AI and machine learning models
* **Automation**: automating tasks and processes
* **Scientific computing**: performing scientific simulations and calculations
* **Education**: teaching programming concepts and fundamentals

Python is known for its:

* **Easy-to-learn syntax**: simple and intuitive syntax makes it accessible to beginners
* **Large community**: extensive community support and resources
* **Extensive libraries**: vast collection of libraries and frameworks for various tasks
* **Cross-platform**: can run on multiple operating systems, including Windows, macOS, and Linux

Some of the key features of Python include:

* **Dynamic typi

## Test 5: Multi-turn Conversation

In [6]:
messages = [
    {"role": "system", "content": "You are a helpful assistant for a cloud billing platform."},
    {"role": "user", "content": "I used 3.5M input tokens and 1.2M output tokens this month. What's my bill?"}
]

# Turn 1
r1 = client.chat.completions.create(model="llama-3.3-70b-versatile", messages=messages, max_tokens=300)
assistant_reply = r1.choices[0].message.content
print(f"Turn 1: {assistant_reply}")
print(f"Tokens: {r1.usage.total_tokens}")

# Turn 2 - follow up
messages.append({"role": "assistant", "content": assistant_reply})
messages.append({"role": "user", "content": "What if I upgrade and my output rate drops to $6/1M?"})

r2 = client.chat.completions.create(model="llama-3.3-70b-versatile", messages=messages, max_tokens=300)
print(f"\nTurn 2: {r2.choices[0].message.content}")
print(f"Tokens: {r2.usage.total_tokens}")

Turn 1: To calculate your bill, I'll need to know the pricing for input and output tokens. Our standard pricing is $0.000004 per input token and $0.000008 per output token. 

For input tokens: 3,500,000 tokens * $0.000004 = $14
For output tokens: 1,200,000 tokens * $0.000008 = $9.60

Your total bill for this month would be $14 + $9.60 = $23.60.
Tokens: 178

Turn 2: If you upgrade and the output rate drops to $6 per 1 million output tokens, that's $0.000006 per output token.

For input tokens, the price remains the same: 3,500,000 tokens * $0.000004 = $14

For output tokens, with the new rate: 1,200,000 tokens * $0.000006 = $7.20

Your new total bill would be $14 + $7.20 = $21.20. Upgrading would save you $2.40 compared to your previous bill.
Tokens: 322


## Test 6: Multiple Rapid Requests (Usage Accumulation)

In [7]:
total_prompt = 0
total_completion = 0

prompts = [
    "What is DynamoDB?",
    "Explain AWS Lambda in one sentence.",
    "What is API Gateway?",
    "Explain Cognito briefly.",
    "What is S3?",
]

for i, prompt in enumerate(prompts):
    r = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=50,
    )
    total_prompt += r.usage.prompt_tokens
    total_completion += r.usage.completion_tokens
    print(f"  [{i+1}] {prompt[:40]:40s} → {r.usage.total_tokens} tokens")

print(f"\n=== TOTAL: prompt={total_prompt}, completion={total_completion}, total={total_prompt + total_completion} ===")
print("\nCheck your dashboard to see these tokens reflected in the usage chart.")

  [1] What is DynamoDB?                        → 90 tokens
  [2] Explain AWS Lambda in one sentence.      → 83 tokens
  [3] What is API Gateway?                     → 90 tokens
  [4] Explain Cognito briefly.                 → 91 tokens
  [5] What is S3?                              → 90 tokens

=== TOTAL: prompt=204, completion=240, total=444 ===

Check your dashboard to see these tokens reflected in the usage chart.


## Interactive Chat — Enter Your Own Prompts

Run the cell below to start an interactive chat session. Type your prompts and get AI responses in real-time. Type `quit` or `exit` to stop.

In [8]:
conversation = [
    {"role": "system", "content": "You are a helpful AI assistant powered by NeuroRouter."}
]
total_tokens_used = 0

print("=== NeuroRouter Interactive Chat ===")
print("Type your message and press Enter. Type 'quit' to exit.\n")

while True:
    user_input = input("You: ")
    if user_input.strip().lower() in ("quit", "exit", "q"):
        print(f"\n--- Session ended. Total tokens used: {total_tokens_used} ---")
        break
    
    conversation.append({"role": "user", "content": user_input})
    
    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=conversation,
            max_tokens=500,
        )
        
        reply = response.choices[0].message.content
        tokens = response.usage.total_tokens
        total_tokens_used += tokens
        
        conversation.append({"role": "assistant", "content": reply})
        
        print(f"\nAI: {reply}")
        print(f"   [{tokens} tokens | session total: {total_tokens_used}]\n")
        
    except Exception as e:
        print(f"\nError: {e}\n")
        conversation.pop()  # remove failed user message

=== NeuroRouter Interactive Chat ===
Type your message and press Enter. Type 'quit' to exit.


--- Session ended. Total tokens used: 0 ---


## Test 7: Verify Dashboard Shows Usage

After running the tests above, open these pages to verify:

1. **http://localhost:3000/dashboard** → Total Tokens and Total Requests should increase
2. **http://localhost:3000/dashboard/usage** → Usage chart should show tokens for this month
3. **http://localhost:3000/dashboard/billing** → Current cycle should show token counts and estimated cost

## Demo: Token Limit Warning Banners

Run the cells below to simulate usage levels and see the warning banners on the dashboard.
- **Cell A**: Sets usage to 85% (Yellow warning — "Approaching Free Tier Limit")
- **Cell B**: Sets usage to 120% (Red warning — "Free Tier Limit Exceeded")

After running each cell, refresh `http://localhost:3000/dashboard` to see the banner.

In [9]:
# DEMO CELL A: Set usage to 85% — triggers YELLOW warning banner
# "Approaching Free Tier Limit" with amber progress bars
import boto3, requests

dynamodb = boto3.client('dynamodb', region_name='ap-south-1')

# Login to get user ID
login = requests.post(f"{API_BASE}/auth/login", json={"email": "test3@gmail.com", "password": "Abc123456789"})
if login.status_code != 200 or "access_token" not in login.json():
    raise Exception(f"Login failed: {login.text}")
jwt_token = login.json()["access_token"]
me = requests.get(f"{API_BASE}/auth/me", headers={"Authorization": f"Bearer {jwt_token}"})
user_id = me.json()["userId"]
print(f"User: {me.json()['email']} (ID: {user_id})")

# Set input to 850k (85%) and output to 750k (75%)
dynamodb.put_item(
    TableName='neurorouter-usage-monthly-dev',
    Item={
        'userId': {'S': user_id},
        'sk': {'S': '2026-04#MODEL#llama-3.3-70b-versatile#KEY#demo-limit-test'},
        'year_month': {'S': '2026-04'},
        'model': {'S': 'llama-3.3-70b-versatile'},
        'api_key_id': {'S': 'demo-limit-test'},
        'input_tokens': {'N': '850000'},
        'output_tokens': {'N': '750000'},
        'total_tokens': {'N': '1600000'},
        'request_count': {'N': '500'},
    }
)

print("\n--- Usage set to 85% input / 75% output ---")
print("   Input:  850,000 / 1,000,000 (85%) -> YELLOW")
print("   Output: 750,000 / 1,000,000 (75%) -> normal")
print("\n>>> Refresh http://localhost:3000/dashboard to see the YELLOW warning banner")


User: test3@gmail.com (ID: 31f38d1a-5071-70ac-3751-c976949af270)

--- Usage set to 85% input / 75% output ---
   Input:  850,000 / 1,000,000 (85%) -> YELLOW
   Output: 750,000 / 1,000,000 (75%) -> normal

>>> Refresh http://localhost:3000/dashboard to see the YELLOW warning banner


In [10]:
# DEMO CELL B: Set usage to 120% — triggers RED exceeded banner
# "Free Tier Limit Exceeded" with red progress bars and overage message
import boto3, requests

dynamodb = boto3.client('dynamodb', region_name='ap-south-1')

# Login to get user ID
login = requests.post(f"{API_BASE}/auth/login", json={"email": "test3@gmail.com", "password": "Abc123456789"})
if login.status_code != 200 or "access_token" not in login.json():
    raise Exception(f"Login failed: {login.text}")
jwt_token = login.json()["access_token"]
me = requests.get(f"{API_BASE}/auth/me", headers={"Authorization": f"Bearer {jwt_token}"})
user_id = me.json()["userId"]
print(f"User: {me.json()['email']} (ID: {user_id})")

dynamodb.put_item(
    TableName='neurorouter-usage-monthly-dev',
    Item={
        'userId': {'S': user_id},
        'sk': {'S': '2026-04#MODEL#llama-3.3-70b-versatile#KEY#demo-limit-test'},
        'year_month': {'S': '2026-04'},
        'model': {'S': 'llama-3.3-70b-versatile'},
        'api_key_id': {'S': 'demo-limit-test'},
        'input_tokens': {'N': '1200000'},
        'output_tokens': {'N': '1100000'},
        'total_tokens': {'N': '2300000'},
        'request_count': {'N': '800'},
    }
)

print("--- Usage set to 120% input / 110% output ---")
print("   Input:  1,200,000 / 1,000,000 (120%) -> RED")
print("   Output: 1,100,000 / 1,000,000 (110%) -> RED")
print("   Overage: +200k input ($0.40) + 100k output ($0.80)")
print("\n>>> Refresh http://localhost:3000/dashboard to see the RED exceeded banner")


User: test3@gmail.com (ID: 31f38d1a-5071-70ac-3751-c976949af270)
--- Usage set to 120% input / 110% output ---
   Input:  1,200,000 / 1,000,000 (120%) -> RED
   Output: 1,100,000 / 1,000,000 (110%) -> RED
   Overage: +200k input ($0.40) + 100k output ($0.80)

>>> Refresh http://localhost:3000/dashboard to see the RED exceeded banner


In [11]:
# DEMO CELL C: Reset usage back to normal (100k tokens)
# Clears warning banners from the dashboard
import boto3, requests

dynamodb = boto3.client('dynamodb', region_name='ap-south-1')

# Login to get user ID
login = requests.post(f"{API_BASE}/auth/login", json={"email": "test3@gmail.com", "password": "Abc123456789"})
if login.status_code != 200 or "access_token" not in login.json():
    raise Exception(f"Login failed: {login.text}")
jwt_token = login.json()["access_token"]
me = requests.get(f"{API_BASE}/auth/me", headers={"Authorization": f"Bearer {jwt_token}"})
user_id = me.json()["userId"]
print(f"User: {me.json()['email']} (ID: {user_id})")

# Delete all existing usage rows for this user
scan = dynamodb.query(
    TableName='neurorouter-usage-monthly-dev',
    KeyConditionExpression='userId = :uid',
    ExpressionAttributeValues={':uid': {'S': user_id}}
)
for item in scan.get('Items', []):
    dynamodb.delete_item(
        TableName='neurorouter-usage-monthly-dev',
        Key={'userId': item['userId'], 'sk': item['sk']}
    )
print(f"Deleted {len(scan.get('Items', []))} existing usage rows")

# Insert a single clean row with 100k tokens
dynamodb.put_item(
    TableName='neurorouter-usage-monthly-dev',
    Item={
        'userId': {'S': user_id},
        'sk': {'S': '2026-04#MODEL#llama-3.3-70b-versatile#KEY#demo-limit-test'},
        'year_month': {'S': '2026-04'},
        'model': {'S': 'llama-3.3-70b-versatile'},
        'api_key_id': {'S': 'demo-limit-test'},
        'input_tokens': {'N': '0'},
        'output_tokens': {'N': '0'},
        'total_tokens': {'N': '0'},
        'request_count': {'N': '50'},
    }
)

print("\n--- Usage reset to 100k tokens ---")
print("   Input:  50,000 / 1,000,000 (5%) -> normal")
print("   Output: 50,000 / 1,000,000 (5%) -> normal")
print("\n>>> Refresh http://localhost:3000/dashboard — no warning banner")


User: test3@gmail.com (ID: 31f38d1a-5071-70ac-3751-c976949af270)
Deleted 1 existing usage rows

--- Usage reset to 100k tokens ---
   Input:  50,000 / 1,000,000 (5%) -> normal
   Output: 50,000 / 1,000,000 (5%) -> normal

>>> Refresh http://localhost:3000/dashboard — no warning banner
